# Hyperliquid API Explorer

Notebook for testing and exploring the Hyperliquid DEX public API.
All requests go to `POST https://api.hyperliquid.xyz/info` — no auth required.

In [1]:
import sys
sys.path.insert(0, '.')

import time
import pandas as pd
import json
from hyperliquid_api import (
    Hyperliquid,
    parse_ob,
    parse_candle,
    coin_context,
    parse_predicted_fundings,
    deal_imbalance,
)

api = Hyperliquid()
COIN = "BTC"   # change to "ETH" or "SOL"
print(f"API ready — testing with {COIN}")

API ready — testing with BTC


## 1. Market Overview — all assets (metaAndAssetCtxs)

In [2]:
meta_ctxs = api.meta_and_contexts()
meta, ctxs = meta_ctxs

rows = []
for i, asset in enumerate(meta["universe"]):
    ctx = ctxs[i]
    rows.append({
        "coin":          asset["name"],
        "mark_price":    float(ctx["markPx"]),
        "oracle_price":  float(ctx["oraclePx"]),
        "fund_rate":     float(ctx["funding"]),
        "premium":       float(ctx.get("premium") or 0),
        "oi_usd":        float(ctx["openInterest"]) * float(ctx["markPx"]),
        "day_volume_usd": float(ctx["dayNtlVlm"]),
    })

df_market = pd.DataFrame(rows).sort_values("oi_usd", ascending=False).reset_index(drop=True)
pd.set_option("display.float_format", "{:,.2f}".format)
df_market.head(20)

,coin,mark_price,oracle_price,fund_rate,premium,oi_usd,day_volume_usd
0,BTC,"80,817.00","80,856.00",0.00,-0.00,"2,289,258,882.60","2,582,190,877.08"
1,ETH,"2,314.10","2,315.20",0.00,-0.00,"1,231,353,132.83","897,997,417.88"
2,HYPE,40.95,40.96,0.00,0.00,"831,021,872.40","254,236,332.32"
3,SOL,94.78,94.81,0.00,0.00,"415,166,792.97","472,999,278.17"
4,ZEC,549.65,549.80,-0.00,0.00,"218,698,864.38","240,134,322.53"
5,XRP,1.47,1.47,0.00,0.00,"95,867,282.05","113,891,388.08"
6,TON,2.30,2.30,0.00,0.00,"92,960,977.18","72,732,195.68"
7,ASTER,0.68,0.68,0.00,0.00,"71,474,098.51","8,312,501.54"
8,DOGE,0.11,0.11,0.00,-0.00,"58,524,947.02","45,411,811.24"
9,AAVE,99.60,99.54,0.00,0.00,"56,560,253.68","33,633,367.35"


## 2. Order Book (nSigFigs)

`nSigFigs` controls price aggregation granularity:
- `None` — full precision, individual tick levels (~20 per side)
- `5` — round to 5 significant figures; `mantissa` (1/2/5) sub-rounds further
- `4` — round to 4 sig figs (wider buckets, fewer levels)
- `3` / `2` — progressively coarser (useful for sparse/regime-level views)

In [12]:
# Full precision (default) — nSigFigs=None
raw_ob = api.orderbook(COIN,n_sig_figs=3)
bids, asks = parse_ob(raw_ob)

mid        = (bids.index[0] + asks.index[0]) / 2
spread_bps = (asks.index[0] - bids.index[0]) / bids.index[0] * 10_000
span_pct   = min(bids.index[0] - bids.index[-1], asks.index[-1] - asks.index[0]) / mid * 100

print(f"nSigFigs=None (full precision)")
print(f"Best bid: {bids.index[0]:,.4f}   Best ask: {asks.index[0]:,.4f}")
print(f"Spread:   {spread_bps:.4f} bps")
print(f"OB span:  {span_pct:.4f}%   levels: {len(bids)} bids / {len(asks)} asks")
print()

top_bids = bids.head(10).reset_index()[["price","size","amount"]].rename(
    columns={"price":"bid_px","size":"bid_sz","amount":"bid_usd"})
top_asks = asks.head(10).reset_index()[["price","size","amount"]].rename(
    columns={"price":"ask_px","size":"ask_sz","amount":"ask_usd"})
pd.concat([top_bids, top_asks], axis=1)

nSigFigs=None (full precision)
Best bid: 81,200.0000   Best ask: 81,300.0000
Spread:   12.3153 bps
OB span:  2.3385%   levels: 20 bids / 20 asks



,bid_px,bid_sz,bid_usd,ask_px,ask_sz,ask_usd
0,"81,200.00",168.90,"13,714,634.53","81,300.00",193.22,"15,708,874.62"
1,"81,100.00",403.66,"32,736,889.26","81,400.00",391.60,"31,876,293.72"
2,"81,000.00",257.21,"20,834,203.59","81,500.00",175.40,"14,295,445.56"
3,"80,900.00",265.72,"21,496,478.60","81,600.00",223.01,"18,197,848.56"
4,"80,800.00",167.36,"13,522,735.67","81,700.00",98.14,"8,017,801.07"
5,"80,700.00",34.91,"2,817,112.72","81,800.00",80.45,"6,580,687.30"
6,"80,600.00",234.51,"18,901,325.46","81,900.00",202.16,"16,557,086.64"
7,"80,500.00",115.23,"9,276,227.52","82,000.00",118.88,"9,748,321.54"
8,"80,400.00",105.27,"8,463,491.72","82,100.00",55.26,"4,536,514.32"
9,"80,300.00",26.61,"2,136,405.59","82,200.00",5.43,"446,212.01"


In [4]:
# nSigFigs comparison — always 20 levels returned; parameter changes price bucket width
#
# nSigFigs=None : raw tick precision  (e.g. $0.1 steps for ETH)
# nSigFigs=5    : 5 sig figs          (similar to None for most prices)
# mantissa=5    : round to .0 / .5    ($0.5 steps for ETH)
# mantissa=2    : round to .0 / .2/.4 ($0.2 steps for ETH)
# nSigFigs=4    : $1 steps for ETH, $10 steps for BTC
# nSigFigs=3    : $10 steps for ETH, $100 steps for BTC
# nSigFigs=2    : $100 steps for ETH, $1000 steps for BTC
# Note: mantissa=1 is documented but returns 500 — skip it

configs = [
    {"n_sig_figs": None, "mantissa": None, "label": "None (raw precision)"},
    {"n_sig_figs": 5,    "mantissa": None, "label": "5"},
    {"n_sig_figs": 5,    "mantissa": 5,    "label": "5 + mantissa=5"},
    {"n_sig_figs": 5,    "mantissa": 2,    "label": "5 + mantissa=2"},
    {"n_sig_figs": 4,    "mantissa": None, "label": "4"},
    {"n_sig_figs": 3,    "mantissa": None, "label": "3"},
    {"n_sig_figs": 2,    "mantissa": None, "label": "2"},
]

rows = []
for cfg in configs:
    raw = api.orderbook(COIN, n_sig_figs=cfg["n_sig_figs"], mantissa=cfg["mantissa"])
    b, a = parse_ob(raw)
    m = (b.index[0] + a.index[0]) / 2
    # bucket width = price step between adjacent bid levels
    bucket = round(b.index[0] - b.index[1], 6) if len(b) > 1 else None
    rows.append({
        "nSigFigs":      cfg["label"],
        "levels":        len(b),
        "bucket_width":  bucket,
        "best_bid":      b.index[0],
        "best_ask":      a.index[0],
        "spread_bps":    round((a.index[0] - b.index[0]) / b.index[0] * 10_000, 4),
        "total_bid_usd": int(b["amount"].sum()),
        "total_ask_usd": int(a["amount"].sum()),
    })

pd.DataFrame(rows)

,nSigFigs,levels,bucket_width,best_bid,best_ask,spread_bps,total_bid_usd,total_ask_usd
0,None (raw precision),20,1.00,"80,895.00","80,896.00",0.12,6695799,432637
1,5,20,1.00,"80,896.00","80,897.00",0.12,15142133,523930
2,5 + mantissa=5,20,5.00,"80,910.00","80,915.00",0.62,28659374,34257465
3,5 + mantissa=2,20,4.00,"80,912.00","80,914.00",0.25,5652118,14017193
4,4,20,10.00,"80,900.00","80,920.00",2.47,47177183,72580902
5,3,20,100.00,"80,900.00","81,000.00",12.36,207075595,201614728
6,2,20,"1,000.00","80,000.00","81,000.00",125.00,366793813,346655499


In [5]:
# Inspect aggregated book at a chosen nSigFigs — top 10 levels side by side
N_SIG_FIGS = 5   # try 2, 3, 4, 5, or None
MANTISSA   = None  # 1, 2, or 5  (only applies when N_SIG_FIGS=5)

raw_agg = api.orderbook(COIN, n_sig_figs=N_SIG_FIGS, mantissa=MANTISSA)
b_agg, a_agg = parse_ob(raw_agg)

print(f"nSigFigs={N_SIG_FIGS}  mantissa={MANTISSA}  —  {len(b_agg)} bid levels / {len(a_agg)} ask levels")

top_bids = b_agg.head(10).reset_index()[["price","size","amount"]].rename(
    columns={"price":"bid_px","size":"bid_sz","amount":"bid_usd"})
top_asks = a_agg.head(10).reset_index()[["price","size","amount"]].rename(
    columns={"price":"ask_px","size":"ask_sz","amount":"ask_usd"})
pd.concat([top_bids, top_asks], axis=1)

nSigFigs=5  mantissa=None  —  20 bid levels / 20 ask levels


,bid_px,bid_sz,bid_usd,ask_px,ask_sz,ask_usd
0,"80,903.00",1.07,"86,713.45","80,904.00",27.35,"2,213,096.56"
1,"80,902.00",0.02,"2,013.65","80,905.00",4.99,"403,555.76"
2,"80,901.00",0.00,38.02,"80,906.00",3.43,"277,275.38"
3,"80,900.00",0.00,13.75,"80,907.00",0.92,"74,437.68"
4,"80,899.00",0.00,13.75,"80,908.00",1.98,"160,145.25"
5,"80,898.00",0.05,"3,964.81","80,909.00",2.17,"175,859.76"
6,"80,897.00",0.00,280.71,"80,910.00",1.67,"135,193.33"
7,"80,896.00",0.96,"77,484.62","80,911.00",7.67,"620,252.40"
8,"80,895.00",0.44,"35,572.77","80,912.00",0.02,"1,879.59"
9,"80,894.00",0.02,"2,013.45","80,913.00",2.37,"191,591.47"


In [6]:
# Cumulative depth profile
depth_levels = [0.01, 0.02, 0.05]
rows = []
for pct in depth_levels:
    if bids.index[0] >= mid:
        b_mask = bids.index >= mid * (1 - pct/100)
        a_mask = asks.index <= mid * (1 + pct/100)
    else:
        b_mask = bids.index >= mid * (1 - pct/100)
        a_mask = asks.index <= mid * (1 + pct/100)
    bid_cum = float(bids.loc[b_mask, "amount"].sum())
    ask_cum = float(asks.loc[a_mask, "amount"].sum())
    rows.append({"level_%": pct, "bid_cum_usd": bid_cum, "ask_cum_usd": ask_cum,
                 "bid/ask": bid_cum / (ask_cum + 1e-12)})

pd.DataFrame(rows)

,level_%,bid_cum_usd,ask_cum_usd,bid/ask
0,0.01,"924,524.07","848,145.50",1.09
1,0.02,"1,862,796.37","2,306,548.03",0.81
2,0.05,"2,155,551.65","2,989,143.79",0.72


## 3. OHLCV Candles

In [7]:
raw_candles = api.candles(COIN, interval="1m", lookback_ms=60 * 60_000)  # last 60 min

df_candles = pd.DataFrame(raw_candles)
df_candles["time"] = pd.to_datetime(df_candles["t"], unit="ms")
df_candles = df_candles.rename(columns={"o":"open","h":"high","l":"low","c":"close","v":"volume","n":"num_trades"})
df_candles[["time","open","high","low","close","volume","num_trades"]].tail(10)

,time,open,high,low,close,volume,num_trades
51,2026-05-11 14:44:00,80798.0,80816.0,80758.0,80765.0,34.33624,408
52,2026-05-11 14:45:00,80765.0,80780.0,80725.0,80725.0,99.00357,564
53,2026-05-11 14:46:00,80726.0,80755.0,80676.0,80753.0,17.24282,555
54,2026-05-11 14:47:00,80755.0,80760.0,80727.0,80727.0,4.74751,317
55,2026-05-11 14:48:00,80728.0,80759.0,80693.0,80725.0,3.58102,315
56,2026-05-11 14:49:00,80726.0,80726.0,80689.0,80724.0,9.82276,255
57,2026-05-11 14:50:00,80725.0,80739.0,80705.0,80734.0,6.96232,265
58,2026-05-11 14:51:00,80730.0,80819.0,80724.0,80818.0,9.08375,518
59,2026-05-11 14:52:00,80818.0,80828.0,80792.0,80819.0,9.37901,284
60,2026-05-11 14:53:00,80819.0,80914.0,80819.0,80903.0,14.52154,593


In [8]:
last_closed = parse_candle(raw_candles)
print("Last closed 1m bar:")
for k, v in last_closed.items():
    print(f"  {k:<12} {v}")

Last closed 1m bar:
  open         80818.0
  high         80828.0
  low          80792.0
  close        80819.0
  volume       9.37901
  num_trades   284


## 4. Recent Trades & Deal Imbalance

In [9]:
raw_trades = api.recent_trades(COIN)

df_trades = pd.DataFrame(raw_trades)
df_trades["time"] = pd.to_datetime(df_trades["time"], unit="ms")
df_trades["px"]   = df_trades["px"].astype(float)
df_trades["sz"]   = df_trades["sz"].astype(float)
df_trades["usd"]  = df_trades["px"] * df_trades["sz"]
df_trades["side_label"] = df_trades["side"].map({"B": "buy", "A": "sell"})

print(f"Trades returned: {len(df_trades)}")
df_trades[["time","px","sz","usd","side_label"]].tail(10)

Trades returned: 10


,time,px,sz,usd,side_label
0,2026-05-11 14:53:44.995,"80,903.00",0.08,"6,294.25",buy
1,2026-05-11 14:53:44.551,"80,903.00",0.00,28.32,sell
2,2026-05-11 14:53:44.551,"80,903.00",0.00,9.71,sell
3,2026-05-11 14:53:44.370,"80,903.00",0.00,14.56,sell
4,2026-05-11 14:53:43.425,"80,904.00",0.00,10.52,sell
5,2026-05-11 14:53:43.425,"80,904.00",0.00,13.75,sell
6,2026-05-11 14:53:43.425,"80,905.00",0.00,24.27,sell
7,2026-05-11 14:53:43.425,"80,905.00",0.00,13.75,sell
8,2026-05-11 14:53:43.310,"80,906.00",0.08,"6,495.13",buy
9,2026-05-11 14:53:43.310,"80,906.00",0.27,"21,545.27",buy


In [10]:
sell_usd, buy_usd = deal_imbalance(raw_trades, lookback_ms=60_000)
ratio = sell_usd / (buy_usd + 1e-12)

print(f"Last 60s taker flow:")
print(f"  sell_usd  ${sell_usd:>12,.0f}")
print(f"  buy_usd   ${buy_usd:>12,.0f}")
print(f"  ratio     {ratio:.4f}  ({'sell-heavy' if ratio > 1 else 'buy-heavy'})")

Last 60s taker flow:
  sell_usd  $         115
  buy_usd   $      34,335
  ratio     0.0033  (buy-heavy)


## 5. Funding Rate — current + predicted cross-venue

In [11]:
ctx = coin_context(meta_ctxs, COIN)
print(f"Current funding rate : {float(ctx['funding']):.6f}  (hourly)")
print(f"Premium index        : {float(ctx.get('premium') or 0):.6f}")
print(f"Impact bid px        : {float((ctx.get('impactPxs') or [0,0])[0]):,.2f}")
print(f"Impact ask px        : {float((ctx.get('impactPxs') or [0,0])[1]):,.2f}")
print()

Current funding rate : 0.000004  (hourly)
Premium index        : -0.000507
Impact bid px        : 80,874.00
Impact ask px        : 80,875.00



In [12]:
raw_pred = api.predicted_fundings()
pred = parse_predicted_fundings(raw_pred, COIN)

print(f"Predicted funding rates for {COIN}:")
print(f"  HL (next period)  : {pred['hl_predicted']:.6f}")
print(f"  Binance (current) : {pred['binance_funding']:.6f}")
print(f"  Bybit   (current) : {pred['bybit_funding']:.6f}")
print()

hl = pred['hl_predicted']
bn = pred['binance_funding']
by = pred['bybit_funding']
print(f"Cross-venue spread vs HL:")
print(f"  Binance - HL : {(bn - hl)*1e4:+.2f} bps/hr")
print(f"  Bybit   - HL : {(by - hl)*1e4:+.2f} bps/hr")

Predicted funding rates for BTC:
  HL (next period)  : 0.000004
  Binance (current) : 0.000029
  Bybit   (current) : 0.000016

Cross-venue spread vs HL:
  Binance - HL : +0.26 bps/hr
  Bybit   - HL : +0.13 bps/hr


## 6. OI Cap Status

In [13]:
at_cap = api.perps_at_oi_cap() or []
print(f"Assets currently at OI cap ({len(at_cap)}): {at_cap if at_cap else 'none'}")
print(f"{COIN} at cap: {COIN in at_cap}")

Assets currently at OI cap (10): ['CANTO', 'FTM', 'HEMI', 'JELLY', 'LOOM', 'RLB', 'SAGA', 'STBL', 'TST', 'ZEREBRO']
BTC at cap: False


## 7. Full 1-minute snapshot row (as stored in DB)

In [14]:
DEPTH_LEVELS_PCT = [0.01, 0.02, 0.05]

def _tag(p): return f'{p*100:05.1f}'.replace('.','').lstrip('0').zfill(3)
def cum_depth(side, mid, pct):
    if side.index[0] >= mid: mask = side.index <= mid * (1 + pct/100)
    else:                     mask = side.index >= mid * (1 - pct/100)
    return float(side.loc[mask, "amount"].sum())
def microprice(b, a):
    bb, ba = float(b.index[0]), float(a.index[0])
    bsz, asz = float(b["size"].iloc[0]), float(a["size"].iloc[0])
    return (bb*asz + ba*bsz) / (bsz+asz) if (bsz+asz) > 0 else (bb+ba)/2
def wall(side, n=100):
    top = side.head(n)
    if top.empty: return 0.0, 0.0
    idx = top["amount"].idxmax()
    return float(idx), float(top.loc[idx, "size"])
def ob_span(b, a): return float(min(b.index[0]-b.index[-1], a.index[-1]-a.index[0]))
def _imb(b, a):
    tb, ta = b["amount"].sum(), a["amount"].sum()
    return (tb-ta) / (tb+ta+1e-12)

lrg = (bids["amount"].sum() + asks["amount"].sum()) * 0.01
wbp, wbs = wall(bids); wap, was_ = wall(asks)

row = {"timestamp": int(time.time())}
for pct in DEPTH_LEVELS_PCT:
    tag = _tag(pct)
    row[f"bid_cum_{tag}pct"] = cum_depth(bids, mid, pct)
    row[f"ask_cum_{tag}pct"] = cum_depth(asks, mid, pct)
row.update({"ask_price": float(asks.index[0]), "bid_price": float(bids.index[0]),
            "mark_price": float(ctx["markPx"]), "oracle_price": float(ctx["oraclePx"]),
            "spread_bps": spread_bps, "ob_depth_span": ob_span(bids, asks),
            "oi_usd": float(ctx["openInterest"])*float(ctx["markPx"]),
            "fund_rate": float(ctx["funding"]),
            "premium": float(ctx.get("premium") or 0),
            "impact_bid_px": float((ctx.get("impactPxs") or [0,0])[0]),
            "impact_ask_px": float((ctx.get("impactPxs") or [0,0])[1]),
            "day_volume_usd": float(ctx["dayNtlVlm"]),
            "predicted_funding_hl":      pred["hl_predicted"],
            "predicted_funding_binance": pred["binance_funding"],
            "predicted_funding_bybit":   pred["bybit_funding"],
            "at_oi_cap": int(COIN in at_cap)})
row.update({"open": last_closed["open"], "high": last_closed["high"],
            "low": last_closed["low"],  "close": last_closed["close"],
            "volume": last_closed["volume"], "num_trades": last_closed["num_trades"]})
row.update({"imbalance": _imb(bids, asks),
            "bid_concentration": bids["amount"].head(10).sum() / (bids["amount"].sum()+1e-12),
            "ask_concentration": asks["amount"].head(10).sum() / (asks["amount"].sum()+1e-12),
            "large_bid_count": int((bids["amount"] > lrg).sum()),
            "large_ask_count": int((asks["amount"] > lrg).sum()),
            "microprice": microprice(bids, asks),
            "wall_bid_price": wbp, "wall_bid_size": wbs,
            "wall_ask_price": wap, "wall_ask_size": was_})
row.update({"sell_usd": sell_usd, "buy_usd": buy_usd,
            "sell_buy_ratio": sell_usd / (buy_usd + 1e-12)})

df_row = pd.DataFrame(row.items(), columns=["column", "value"])
print(f"Total columns: {len(row)}")
df_row

Total columns: 42


,column,value
0,timestamp,"1,778,511,227.00"
1,bid_cum_010pct,"924,524.07"
2,ask_cum_010pct,"848,145.50"
3,bid_cum_020pct,"1,862,796.37"
4,ask_cum_020pct,"2,306,548.03"
5,bid_cum_050pct,"2,155,551.65"
6,ask_cum_050pct,"2,989,143.79"
7,ask_price,"80,896.00"
8,bid_price,"80,895.00"
9,mark_price,"80,880.00"
